<a href="https://colab.research.google.com/github/onzl-c/Hongik_AIMLproject/blob/learning/ensemble_lightgbm_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, Dropout
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
colab_path = 'drive/MyDrive/Colab Notebooks/detection/binary_2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load datasets
data = pd.read_csv(colab_path + '/final_dataset.csv')
verification_dataset = pd.read_excel(colab_path + '/verification.csv')

In [ ]:
data['Y'].value_counts()

,count
Y,
1.0,34764
0.0,12605


In [ ]:
# Identify columns with object (string) dtype
categorical_cols = data.select_dtypes(include=['object']).columns

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Iterate through categorical columns and encode them
for col in categorical_cols:
    data[col] = label_encoder.fit_transform(data[col])

# Handle missing values by filling NaNs with column mean
data = data.fillna(data.mean())

# Ensure that the target variable 'Y' is also numeric
data['Y'] = pd.to_numeric(data['Y'], errors='coerce').fillna(0).astype(int)

In [ ]:
data['Y'].value_counts()

,count
Y,
1,34764
0,12598


In [ ]:
# Separate features and labels
# X = data.drop('Y', axis=1).values
# y = data['Y'].values

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# LightGBM model
lgb_model = lgb.LGBMClassifier()
lgb_model.fit(X_train, y_train)
lgb_predictions = lgb_model.predict(X_test)
lgb_accuracy = accuracy_score(y_test, lgb_predictions)
print(f"LightGBM Accuracy: {lgb_accuracy:.4f}")

[LightGBM] [Info] Number of positive: 27766, number of negative: 10123
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2753
[LightGBM] [Info] Number of data points in the train set: 37889, number of used features: 77
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.732825 -> initscore=1.009002
[LightGBM] [Info] Start training from score 1.009002
LightGBM Accuracy: 0.9195


In [ ]:
# Keras model
keras_model = Sequential()
keras_model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
keras_model.add(Dropout(0.5))
keras_model.add(Dense(32, activation='relu'))
keras_model.add(Dense(1, activation='sigmoid'))
keras_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)
keras_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32, callbacks=[early_stopping], verbose=0)
keras_accuracy = keras_model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Keras MLP Accuracy: {keras_accuracy:.4f}")

Epoch 8: early stopping
Keras MLP Accuracy: 0.7404


In [ ]:
# CNN Model
cnn_model = Sequential()
cnn_model.add(Conv1D(32, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
cnn_model.add(Flatten())
cnn_model.add(Dense(16, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))
cnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Reshape X for CNN
X_train_cnn = np.expand_dims(X_train, axis=2)
X_test_cnn = np.expand_dims(X_test, axis=2)

cnn_model.fit(X_train_cnn, y_train, validation_data=(X_test_cnn, y_test), epochs=50, batch_size=32, callbacks=[early_stopping], verbose=0)
cnn_accuracy = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
print(f"CNN Accuracy: {cnn_accuracy:.4f}")

Epoch 35: early stopping
CNN Accuracy: 0.8637


In [ ]:
# Ensemble models (LightGBM + Keras, LightGBM + CNN, LightGBM + CNN + Keras)
class KerasClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model.fit(X, y, epochs=50, batch_size=32, verbose=0)
        return self

    def predict(self, X):
        return (self.model.predict(X) > 0.5).astype(int)

    def predict_proba(self, X):
        proba = self.model.predict(X)
        return np.hstack([1 - proba, proba])

lgb_keras_ensemble = VotingClassifier(estimators=[
    ('lgb', lgb_model),
    ('keras', KerasClassifier(keras_model))
], voting='soft')

lgb_keras_ensemble.fit(X_train, y_train)
lgb_keras_predictions = lgb_keras_ensemble.predict(X_test)
lgb_keras_accuracy = accuracy_score(y_test, lgb_keras_predictions)
print(f"LightGBM + Keras Ensemble Accuracy: {lgb_keras_accuracy:.4f}")

lgb_cnn_ensemble = VotingClassifier(estimators=[
    ('lgb', lgb_model),
    ('cnn', KerasClassifier(cnn_model))
], voting='soft')

lgb_cnn_ensemble.fit(X_train, y_train)
lgb_cnn_predictions = lgb_cnn_ensemble.predict(X_test)
lgb_cnn_accuracy = accuracy_score(y_test, lgb_cnn_predictions)
print(f"LightGBM + CNN Ensemble Accuracy: {lgb_cnn_accuracy:.4f}")

lgb_cnn_keras_ensemble = VotingClassifier(estimators=[
    ('lgb', lgb_model),
    ('cnn', KerasClassifier(cnn_model)),
    ('keras', KerasClassifier(keras_model))
], voting='soft')

lgb_cnn_keras_ensemble.fit(X_train, y_train)
lgb_cnn_keras_predictions = lgb_cnn_keras_ensemble.predict(X_test)
lgb_cnn_keras_accuracy = accuracy_score(y_test, lgb_cnn_keras_predictions)
print(f"LightGBM + CNN + Keras Ensemble Accuracy: {lgb_cnn_keras_accuracy:.4f}")

[LightGBM] [Info] Number of positive: 27766, number of negative: 10123
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065813 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2753
[LightGBM] [Info] Number of data points in the train set: 37889, number of used features: 77
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.732825 -> initscore=1.009002
[LightGBM] [Info] Start training from score 1.009002
297/297 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
LightGBM + Keras Ensemble Accuracy: 0.8995
[LightGBM] [Info] Number of positive: 27766, number of negative: 10123
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020747 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2753
[LightGBM] [Info] Number of d